# Feature Engineering — Smart Attendance System (Biometric IoT Data)

Input: `cleaned_data.csv` (output of `data_cleaning.ipynb`)
Output: `featured_data.csv` — one row per **event** with engineered features, plus `employee_daily.csv` — one row per **employee per day**, which is what the dashboard/ML layer will actually consume.

Two separate outputs because attendance analytics and biometric-reliability analytics operate at different grains: a single Check-In event is one row, but "was this employee present and on time today" is a per-employee-per-day fact built from pairing Check-In + Check-Out.

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/proccesed/cleaned_data.csv')
df['timestamp'] = pd.to_datetime(df['timestamp'])
df['date'] = pd.to_datetime(df['date']).dt.date
df.shape

(10000, 25)

## 1. Time-based features

Basic calendar/time features every downstream model or dashboard filter will need.

In [2]:
df['day_of_week'] = pd.to_datetime(df['date']).dt.day_name()
df['is_weekend'] = pd.to_datetime(df['date']).dt.dayofweek >= 5
df['week_of_year'] = pd.to_datetime(df['date']).dt.isocalendar().week
df['hour'] = df['timestamp'].dt.hour

df[['timestamp','date','day_of_week','is_weekend','week_of_year','hour']].head()

,timestamp,date,day_of_week,is_weekend,week_of_year,hour
0,2026-05-01 03:16:45,2026-05-01,Friday,False,18,3
1,2026-05-01 05:40:46,2026-05-01,Friday,False,18,5
2,2026-05-01 05:46:13,2026-05-01,Friday,False,18,5
3,2026-05-01 05:46:46,2026-05-01,Friday,False,18,5
4,2026-05-01 05:47:13,2026-05-01,Friday,False,18,5


## 2. Biometric reliability features

Per-session features that quantify how "clean" the fingerprint match was — these feed straight into the anomaly-detection layer.

In [3]:
# Binary flags (numeric, model-ready) alongside the existing Yes/No text columns
df['match_success'] = (df['fingerprint_match'] == 'Yes').astype(int)
df['is_multi_attempt'] = (df['attempt_number'] > 1).astype(int)
df['is_unauthorized'] = (df['event_type'] == 'Unauthorized Attempt').astype(int)

# Auth time relative to that device's own typical speed — a slow read on a normally-fast
# device is a stronger signal than comparing against the global average
device_auth_mean = df.groupby('device_id')['authentication_time_ms'].transform('mean')
device_auth_std  = df.groupby('device_id')['authentication_time_ms'].transform('std')
df['auth_time_zscore'] = (df['authentication_time_ms'] - device_auth_mean) / device_auth_std

df[['device_id','fingerprint_match','match_success','attempt_number','is_multi_attempt',
    'authentication_time_ms','auth_time_zscore','is_unauthorized']].head()

,device_id,fingerprint_match,match_success,attempt_number,is_multi_attempt,authentication_time_ms,auth_time_zscore,is_unauthorized
0,GATE-03,No,0,1,0,619.5,2.390153,1
1,GATE-02,Yes,1,1,0,372.4,0.003345,0
2,GATE-02,Yes,1,1,0,391.5,0.196945,0
3,GATE-01,Yes,1,1,0,358.8,-0.171473,0
4,GATE-02,Yes,1,1,0,426.1,0.547655,0


## 3. Device / IoT health features

In [4]:
df['is_device_healthy'] = (df['device_health_score'] >= 90).astype(int)
df['is_weak_signal'] = (df['wifi_rssi_dbm'] < -70).astype(int)
df['has_alert'] = (df['alert_type'] != 'No Alert').astype(int)
df['is_sync_failed'] = (df['database_sync_status'] == 'FAILED').astype(int)

df[['device_id','device_health_score','is_device_healthy','wifi_rssi_dbm','is_weak_signal',
    'alert_type','has_alert','is_sync_failed']].head()

,device_id,device_health_score,is_device_healthy,wifi_rssi_dbm,is_weak_signal,alert_type,has_alert,is_sync_failed
0,GATE-03,96.0,1,-67.6,0,Unknown Fingerprint,1,0
1,GATE-02,96.2,1,-54.0,0,No Alert,0,0
2,GATE-02,93.8,1,-60.5,0,No Alert,0,0
3,GATE-01,95.2,1,-48.0,0,No Alert,0,0
4,GATE-02,95.8,1,-64.9,0,No Alert,0,0


## 4. Unusual check-in time (behavioral anomaly)

For each employee, flag a check-in that falls far outside *their own* usual check-in time — a stronger anomaly signal than comparing against the department/shift average, since it can catch things like a stolen/spoofed fingerprint being used at a time the real employee never shows up.

In [5]:
checkins = df[df['event_type'] == 'Check-In'].copy()
checkins['checkin_minutes'] = checkins['hour'] * 60 + checkins['timestamp'].dt.minute

emp_mean = checkins.groupby('employee_id')['checkin_minutes'].transform('mean')
emp_std  = checkins.groupby('employee_id')['checkin_minutes'].transform('std').replace(0, np.nan)

checkins['checkin_time_zscore'] = (checkins['checkin_minutes'] - emp_mean) / emp_std
checkins['is_unusual_checkin_time'] = (checkins['checkin_time_zscore'].abs() > 2).astype(int)

df = df.merge(
    checkins[['record_id','checkin_time_zscore','is_unusual_checkin_time']],
    on='record_id', how='left'
)

checkins[['employee_id','timestamp','checkin_minutes','checkin_time_zscore','is_unusual_checkin_time']].head()

,employee_id,timestamp,checkin_minutes,checkin_time_zscore,is_unusual_checkin_time
1,E1037,2026-05-01 05:40:46,340,-1.210969,0
2,E1015,2026-05-01 05:46:13,346,-0.905101,0
3,E1008,2026-05-01 05:46:46,346,-0.924300,0
4,E1058,2026-05-01 05:47:13,347,-0.803484,0
5,E1050,2026-05-01 05:50:57,350,-0.386699,0


## 5. Device mismatch (behavioral anomaly)

Flag when an employee checks in at a gate device they rarely/never use — a legitimate employee usually enters through the same gate day to day.

In [6]:
usual_device = (
    checkins.groupby('employee_id')['device_id']
    .agg(lambda x: x.value_counts().idxmax())
    .rename('usual_device')
)

checkins = checkins.merge(usual_device, on='employee_id', how='left')
checkins['is_device_mismatch'] = (checkins['device_id'] != checkins['usual_device']).astype(int)

df = df.merge(
    checkins[['record_id','usual_device','is_device_mismatch']],
    on='record_id', how='left'
)

checkins[checkins['is_device_mismatch']==1][['employee_id','device_id','usual_device']].head()

,employee_id,device_id,usual_device


## 6. Session frequency spikes

Multiple authentication attempts (any event type) for the same employee within a short window can indicate a malfunctioning sensor, or someone repeatedly trying to force a match.

In [7]:
df = df.sort_values(['employee_id', 'timestamp'])
df['prev_event_time'] = df.groupby('employee_id')['timestamp'].shift(1)
df['minutes_since_last_event'] = (df['timestamp'] - df['prev_event_time']).dt.total_seconds() / 60
df['is_rapid_repeat'] = (df['minutes_since_last_event'] < 2).astype(int)

df[['employee_id','timestamp','minutes_since_last_event','is_rapid_repeat']].dropna().head(10)

,employee_id,timestamp,minutes_since_last_event,is_rapid_repeat
127,E1001,2026-05-02 05:56:33,478.050000,0
240,E1001,2026-05-02 22:13:14,976.683333,0
262,E1001,2026-05-03 06:01:02,467.800000,0
346,E1001,2026-05-04 21:40:32,2379.500000,0
414,E1001,2026-05-05 06:08:36,508.066667,0
487,E1001,2026-05-05 21:48:56,940.333333,0
561,E1001,2026-05-06 06:22:21,513.416667,0
647,E1001,2026-05-06 22:16:34,954.216667,0
693,E1001,2026-05-07 06:38:36,502.033333,0
750,E1001,2026-05-07 21:39:59,901.383333,0


## 7. Composite anomaly score

A simple additive score combining the anomaly flags above. Not a model prediction — just a transparent, explainable score for the dashboard to sort/filter by. A proper Isolation Forest model can be trained on top of these same features later if needed.

In [8]:
anomaly_flags = ['is_multi_attempt', 'is_unauthorized', 'is_weak_signal', 'has_alert',
                  'is_sync_failed', 'is_unusual_checkin_time', 'is_device_mismatch', 'is_rapid_repeat']

for col in anomaly_flags:
    df[col] = df[col].fillna(0).astype(int)

df['anomaly_score'] = df[anomaly_flags].sum(axis=1)

df['anomaly_score'].value_counts().sort_index()

anomaly_score
0    5540
1    3240
2     816
3     362
4      38
5       4
Name: count, dtype: int64

## 8. Employee-day level features (Check-In + Check-Out pairing)

This is the table the attendance dashboard and any absenteeism-prediction model should actually be built on — one row per employee per working day.

In [9]:
ci = df[df['event_type']=='Check-In'][['employee_id','department','shift','date','timestamp','is_late']]
ci = ci.rename(columns={'timestamp':'checkin_time'})

co = df[df['event_type']=='Check-Out'][['employee_id','date','timestamp','is_early_exit']]
co = co.rename(columns={'timestamp':'checkout_time'})

daily = ci.merge(co, on=['employee_id','date'], how='left')

# Night-shift checkouts land after midnight, so checkout_time's clock time can be
# *earlier* than checkin_time even though it happened later. Roll checkout forward
# a day whenever that happens, so hours worked doesn't come out negative.
overnight = daily['checkout_time'] < daily['checkin_time']
daily.loc[overnight, 'checkout_time'] = daily.loc[overnight, 'checkout_time'] + pd.Timedelta(days=1)
print(f"Corrected {overnight.sum()} overnight (Night shift) rows")

daily['total_hours_present'] = (
    (daily['checkout_time'] - daily['checkin_time']).dt.total_seconds() / 3600
).round(2)

daily['checked_out'] = daily['checkout_time'].notnull().astype(int)

daily.head()

Corrected 1092 overnight (Night shift) rows


,employee_id,department,shift,date,checkin_time,is_late,checkout_time,is_early_exit,total_hours_present,checked_out
0,E1001,Production,Night,2026-05-01,2026-05-01 21:58:30,No,NaT,NaN,NaN,0
1,E1001,Production,Night,2026-05-02,2026-05-02 22:13:14,Yes,2026-05-03 05:56:33,No,7.72,1
2,E1001,Production,Night,2026-05-04,2026-05-04 21:40:32,No,NaT,NaN,NaN,0
3,E1001,Production,Night,2026-05-05,2026-05-05 21:48:56,No,2026-05-06 06:08:36,No,8.33,1
4,E1001,Production,Night,2026-05-06,2026-05-06 22:16:34,Yes,2026-05-07 06:22:21,No,8.10,1


In [10]:
# Rolling 7-day attendance trend per employee
daily = daily.sort_values(['employee_id','date'])
daily['date'] = pd.to_datetime(daily['date'])

daily['rolling_7d_days_present'] = (
    daily.groupby('employee_id')['date']
    .transform(lambda x: x.rolling(7, min_periods=1).count())
)

# Consecutive absent-day streak needs the full calendar per employee, not just days they showed up.
# Build a full employee x date grid, mark presence, then measure streaks of 0s.
full_range = pd.date_range(daily['date'].min(), daily['date'].max())
employees = daily['employee_id'].unique()
grid = pd.MultiIndex.from_product([employees, full_range], names=['employee_id','date']).to_frame(index=False)

present = daily[['employee_id','date']].drop_duplicates()
present['present'] = 1

grid = grid.merge(present, on=['employee_id','date'], how='left')
grid['present'] = grid['present'].fillna(0).astype(int)

grid['absent_streak_group'] = (grid['present'] != grid.groupby('employee_id')['present'].shift()).cumsum()
grid['consecutive_absent_days'] = grid.groupby(['employee_id','absent_streak_group']).cumcount() + 1
grid.loc[grid['present']==1, 'consecutive_absent_days'] = 0

current_streak = grid.sort_values('date').groupby('employee_id').tail(1)[['employee_id','consecutive_absent_days']]
current_streak = current_streak.rename(columns={'consecutive_absent_days':'current_absent_streak'})

daily = daily.merge(current_streak, on='employee_id', how='left')
daily[['employee_id','date','rolling_7d_days_present','current_absent_streak']].head(10)

,employee_id,date,rolling_7d_days_present,current_absent_streak
0,E1001,2026-05-01,1.0,1
1,E1001,2026-05-02,2.0,1
2,E1001,2026-05-04,3.0,1
3,E1001,2026-05-05,4.0,1
4,E1001,2026-05-06,5.0,1
5,E1001,2026-05-07,6.0,1
6,E1001,2026-05-08,7.0,1
7,E1001,2026-05-11,7.0,1
8,E1001,2026-05-12,7.0,1
9,E1001,2026-05-13,7.0,1


## 9. Cumulative attendance percentage per employee

In [11]:
total_working_days = daily['date'].nunique()
attendance_pct = (daily.groupby('employee_id')['date'].nunique() / total_working_days * 100).round(2)
attendance_pct = attendance_pct.rename('attendance_percentage')

daily = daily.merge(attendance_pct, on='employee_id', how='left')
daily['absentee_risk_flag'] = (daily['attendance_percentage'] < 80).astype(int)

daily[['employee_id','department','attendance_percentage','absentee_risk_flag']].drop_duplicates('employee_id').sort_values('attendance_percentage').head(10)

,employee_id,department,attendance_percentage,absentee_risk_flag
2977,E1044,Maintenance,77.38,1
1880,E1028,Production,78.57,1
3610,E1053,Quality Control,78.57,1
4082,E1060,Packing,78.57,1
2013,E1030,Admin/HR,79.76,1
1050,E1016,Packing,79.76,1
1604,E1024,Production,79.76,1
2080,E1031,Maintenance,79.76,1
3947,E1058,Maintenance,79.76,1
2630,E1039,Quality Control,79.76,1


## 10. Encode categoricals (ML-ready)

Label-encoded copies of the key categorical columns, kept alongside the original text columns so both the dashboard (needs readable labels) and a future ML model (needs numeric input) can use the same file.

In [12]:
from pandas.api.types import CategoricalDtype

for col in ['department', 'shift']:
    daily[col + '_encoded'] = daily[col].astype('category').cat.codes

daily[['department','department_encoded','shift','shift_encoded']].drop_duplicates().head()

,department,department_encoded,shift,shift_encoded
0,Production,3,Night,3
70,Warehouse,6,Afternoon,0
141,Production,3,Afternoon,0
210,Admin/HR,0,General,1
278,Production,3,Morning,2


## 11. Save outputs

In [13]:
df.to_csv('../data/proccesed/featured_data.csv', index=False)
daily.to_csv('../data/proccesed/employee_daily.csv', index=False)

print("Event-level featured data:", df.shape)
print("Employee-day level data  :", daily.shape)
print("\nSaved:")
print(" ../data/proccesed/featured_data.csv")
print(" ../data/proccesed/employee_daily.csv")

Event-level featured data: (10000, 44)
Employee-day level data  : (4148, 16)

Saved:
 ../data/proccesed/featured_data.csv
 ../data/proccesed/employee_daily.csv
